*  So far, we have been working with synthetic data that arrived in ready-made tensors.
*  However, to apply deep learning in the wild we must extract messy data stored in arbitrary formats, and preprocess it to suit our needs. Fortunately, the pandas library can do much of the heavy lifting.
*  This section, while no substitute for a proper pandas tutorial, will give you a crash course on some of the most common routines.


   

# 2.2.1. Reading the Dataset

*   Comma-separated values (CSV) files are ubiquitous for the storing of tabular (spreadsheet-like) data.
*   In them, each line corresponds to one record and consists of several (comma-separated) fields, e.g., “Albert Einstein,March 14 1879,Ulm,Federal polytechnic school,field of gravitational physics”.
*   To demonstrate how to load CSV files with pandas, we create a CSV file below `../data/house_tiny.csv`.
*  This file represents a dataset of homes, where each row corresponds to a distinct home and the columns correspond to the number of rooms (`NumRooms`), the roof type (`RoofType`), and the price (`Price`).

   

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
folder = '/content/drive/MyDrive/Colab Notebooks/Dive into Deep Learning/data/'
os.makedirs(folder, exist_ok=True)
data_file = os.path.join(folder, 'house_tiny.csv')

with open(data_file, "w") as f: #notice how we don't have leading spaces for the entries to prevent the CSV storing numbers like 2 and 4 as strings
  f.write('''NumRooms,RoofType,Price
NA,NA,127500
2,NA,106000
4,Slate,178100
NA,NA,140000''')

Mounted at /content/drive


*   Now let’s import `pandas` and load the dataset with `read_csv`.




In [ ]:
import pandas as pd
data = pd.read_csv(data_file)
print(data)

   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


# 2.2.2. Data Preparation

*   In supervised learning, we train models
to predict a designated *target* value,
given some set of *input* values.
*   Our first step in processing the dataset
is to separate out columns corresponding
to input versus target values.
*   We can select columns either by name or
via integer-location based indexing (`iloc`).


*   You might have noticed that `pandas` replaced
all CSV entries with value `NA`
with a special `NaN` (*not a number*) value.
*   This can also happen whenever an entry is empty,
e.g., $3,,,270000$.
*   These are called *missing values*
and they are the "bed bugs" of data science,
a persistent menace that you will confront
throughout your career.
*   Depending upon the context,
missing values might be handled
either via *imputation* or *deletion*.
> *   *Imputation* replaces missing values with estimates of their values
> *   while *deletion* simply discards either those rows or those columns that contain missing values.

*   Here are some common imputation heuristics.
> *   [**For categorical input fields, we can treat `NaN` as a category.**]
> *   Since the `RoofType` column takes values `Slate` and `NaN`, `pandas` can convert this column into two columns `RoofType_Slate` and `RoofType_nan`.
> *   A row whose roof type is `Slate` will set values of `RoofType_Slate` and `RoofType_nan` to 1 and 0, respectively.
> *   The converse holds for a row with a missing `RoofType` value.


In [ ]:
inputs, targets = data.iloc[:, 0:2], data.iloc[:, 2] #inputs are all of the rows with columns 0 and 1, and targets are all of the rows with column 2
inputs = pd.get_dummies(inputs, dummy_na=True) #inspect all lcolumns in inputs, and convert categorical (strings) into dummies type columns
                                               #dummy_na argument tells the function to treat NaN as their own category and create a separate column for them
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       NaN           False          True
1       2.0           False          True
2       4.0            True         False
3       NaN           False          True


*   For missing numerical values, one common heuristic is to replace the NaN entries with the mean value of the corresponding column.



In [ ]:
inputs = inputs.fillna(inputs.mean()) #converted NaNs in NumRooms to the average of numbers ((2+4)/2)
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       3.0           False          True
1       2.0           False          True
2       4.0            True         False
3       3.0           False          True


# 2.2.3. Conversion to the Tensor Format

*   Now that all the entries in inputs and targets are numerical, we can load them into a tensor (recall Section 2.1).



In [ ]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float)) #PyTorch cannot consume a pandas DataFrame so it has to be converted into NumPy first
y = torch.tensor(targets.to_numpy(dtype=float))
X, y #in data science, cap letters are used for matrix (2D) and small caps are used for vectors (1D)

(tensor([[3., 0., 1.],
         [2., 0., 1.],
         [4., 1., 0.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

# 2.2.4. Discussion

*   You now know how to partition data columns, impute missing variables, and load pandas data into tensors.
*   In Section 5.7, you will pick up some more data processing skills.
*   While this crash course kept things simple, data processing can get hairy.
> * For example, rather than arriving in a single CSV file, our dataset might be spread across multiple files extracted from a relational database.  
> * For instance, in an e-commerce application, customer addresses might live in one table and purchase data in another.
> * Moreover, practitioners face myriad data types beyond categorical and numeric, for example, text strings, images, audio data, and point clouds.
> * Oftentimes, advanced tools and efficient algorithms are required in order to prevent data processing from becoming the biggest bottleneck in the machine learning pipeline.
> * These problems will arise when we get to computer vision and natural language processing.
* Finally, we must pay attention to data quality. Real-world datasets are often plagued by outliers, faulty measurements from sensors, and recording errors, which must be addressed before feeding the data into any model.
* Data visualization tools such as [seaborn](https://https://seaborn.pydata.org/), [Bokeh](https://docs.bokeh.org/en/latest/), or [matplotlib](https://matplotlib.org/) can help you to manually inspect the data and develop intuitions about the type of problems you may need to address.

        

# 2.2.5. Exercises

1. Try loading datasets, e.g., Abalone from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/) and inspect their properties. What fraction of them has missing values? What fraction of the variables is numerical, categorical, or text?

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"

df = pd.read_csv(url) #the abalone dataset doens't have headers
df.head()

,M,0.455,0.365,0.095,0.514,0.2245,0.101,0.15,15
0,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
1,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
2,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
3,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7
4,I,0.425,0.300,0.095,0.3515,0.1410,0.0775,0.120,8


2. Try indexing and selecting data columns by name rather than by column number. The pandas documentation on [indexing](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html) has further details on how to do this.

In [ ]:
df["M"].head() #for columns

,M
0,M
1,F
2,M
3,I
4,I


3. How large a dataset do you think you could load this way? What might be the limitations? Hint: consider the time to read the data, representation, processing, and memory footprint. Try this out on your laptop. What happens if you try it out on a server?
> * You can load datasets via URL using pandas only they comfortably fit into RAM (~< 1–2 GB on a laptop or 16–512 GB on a server)

4. How would you deal with data that has a very large number of categories? What if the category labels are all unique? Should you include the latter?
> * Group rare categories (“bucketting”) e.g. top 100 cities + "other"
> * Frequency encoding e.g. LA 3K
> * Target encoding e.g. City → average house price
> * Embedding (learn a dense vector representation)

5. What alternatives to pandas can you think of? How about [loading NumPy tensors from a file](https://numpy.org/doc/stable/reference/generated/numpy.load.html)? Check out [Pillow](https://pillow.readthedocs.io/en/stable/), the Python Imaging Library.